# Fine-tuned DenseNet-121 Classifier — Pneumonia vs Healthy

Variante do 1.3 com **fine-tuning** do backbone DenseNet-121 (CheXNet-style).  
Descongela `denseblock4 + norm5 + classifier` e treina com **learning rates diferenciados**
(head 1e-3, backbone 1e-5).

Mudanças em relação ao 1.3:
- `unfreeze_from="denseblock4"` no construtor do modelo
- `optimizer = Adam(model.param_groups(head_lr, backbone_lr), weight_decay=...)`
- `horizontal_flip_p = 0.0` (raio-X é anatomicamente assimétrico — coração à esquerda)
- `num_epochs = 30`, `early_stopping_patience = 7`

**Reported metrics:** Accuracy and AUC-ROC on test set.

## 1. Setup

In [ ]:
import sys
import os
import json
from pathlib import Path

project_root = Path("../").resolve()
sys.path.insert(0, str(project_root / "utils"))
sys.path.insert(0, str(project_root / "models"))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from classifier import FrozenDenseNet121
from dataset import AugmentedDataset
from device import DEVICE
from metrics import print_metrics, plot_roc_curve
from train_models import run_epoch, train_model, training_curves, save_results
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

print(f"Device: {DEVICE}")

## 2. Hyperparameters (from YAML)

In [ ]:
from config import load_config

cfg = load_config(project_root / "configs" / "densenet121_finetune.yaml", project_root)

print(cfg.model.name)
print("unfreeze_from :", cfg.model.unfreeze_from)
print("img_size      :", cfg.data.img_size)
print("class_balance :", cfg.data.class_balancing)
print("batch_size    :", cfg.training.batch_size)
print("head_lr       :", cfg.training.head_lr)
print("backbone_lr   :", cfg.training.backbone_lr)
print("num_epochs    :", cfg.training.num_epochs)
print("weight_decay  :", cfg.training.weight_decay)
print("es_patience   :", cfg.training.early_stopping_patience)
print("seed          :", cfg.seed)

if cfg.data.augmentation:
    print("Augmentations:", cfg.data.augmentations)
else:
    print("No data augmentation will be applied.")

print(cfg.checkpoint_path)
print(cfg.results_dir)

In [ ]:
IMG_SIZE     = tuple(cfg.data.img_size)
BATCH_SIZE   = cfg.training.batch_size
HEAD_LR      = cfg.training.head_lr
BACKBONE_LR  = cfg.training.backbone_lr
NUM_EPOCHS   = cfg.training.num_epochs
SEED         = cfg.seed

CHECKPOINT_PATH = cfg.checkpoint_path
RESULTS_DIR     = cfg.results_dir
RESULTS_DIR.mkdir(exist_ok=True)

## 3. Load Dataset (from cache)

In [ ]:
DATA_CACHE_DIR = project_root / "data" / "processed"
train_dataset = torch.load(DATA_CACHE_DIR / "train_dataset.pt", weights_only=False)
val_dataset   = torch.load(DATA_CACHE_DIR / "val_dataset.pt",   weights_only=False)
test_dataset  = torch.load(DATA_CACHE_DIR / "test_dataset.pt",  weights_only=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

## 4. Data Augmentation

Applied **only** to the training set. Same light setup used in 1.2:  
`RandomRotation(10°)` · `RandomHorizontalFlip(0.5)` · `RandomResizedCrop(scale=[0.95, 1.0])`.

`ColorJitter` is intentionally **not** used here — ImageNet-pretrained features expect natural-distribution intensities.

In [ ]:
aug_transform = transforms.Compose([
    transforms.RandomRotation(degrees=cfg.data.augmentations.random_rotation_degrees),
    transforms.RandomHorizontalFlip(p=cfg.data.augmentations.horizontal_flip_p),
    # transforms.ColorJitter(brightness=cfg.data.augmentations.brightness_jitter),  # disabled
    transforms.RandomResizedCrop(size=IMG_SIZE, scale=tuple(cfg.data.augmentations.random_resized_crop_scale)),
])

aug_train_dataset = AugmentedDataset(train_dataset, aug_transform)
print(f"Augmented training set size: {len(aug_train_dataset)}")

## 5. Model — DenseNet-121 com `denseblock4` descongelado

In [ ]:
torch.manual_seed(SEED)
model = FrozenDenseNet121(unfreeze_from=cfg.model.unfreeze_from).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"unfreeze_from       : {cfg.model.unfreeze_from}")
print(f"Trainable parameters: {trainable:,} / {total:,}")

## 6. DataLoaders & pos_weight

In [ ]:
train_loader = DataLoader(aug_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,       batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,      batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

train_labels = train_dataset.labels[:, 1]  # index 1 = pneumonia
n_pos = train_labels.sum().item()
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([np.sqrt(n_neg / n_pos)], dtype=torch.float32).to(DEVICE)

print(f"Train — Healthy: {int(n_neg)} | Pneumonia: {int(n_pos)}")
print(f"Raw ratio: {n_neg/n_pos:.2f}")
print(f"Attenuated pos_weight (sqrt): {pos_weight.item():.2f}")

## 7. Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE)).to(DEVICE)

# LR diferenciado: head treina rápido, backbone descongelado treina devagar.
optimizer = torch.optim.Adam(
    model.param_groups(head_lr=HEAD_LR, backbone_lr=BACKBONE_LR),
    weight_decay=cfg.training.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

for i, g in enumerate(optimizer.param_groups):
    n = sum(p.numel() for p in g['params'])
    print(f"group {i}: {n:>10,} params @ lr={g['lr']}")

## 8. Training Loop

In [ ]:
best_val_auc, history = train_model(
    model,
    train_loader,
    val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    CHECKPOINT_PATH=CHECKPOINT_PATH,
    device=DEVICE,
    early_stopping_patience=cfg.training.early_stopping_patience,
)

## 9. Training Curves

In [ ]:
training_curves(history, results_path=RESULTS_DIR / "training_curves.png")

## 10. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

_, test_y, test_scores = run_epoch(test_loader, model, criterion, device=DEVICE)
_, val_y,  val_sc      = run_epoch(val_loader,  model, criterion, device=DEVICE)

from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(val_y, val_sc)
best_threshold = thresholds[np.argmax(tpr - fpr)]

print("=== Test Set Metrics ===")
print_metrics(test_y, test_scores, threshold=best_threshold)

In [ ]:
fig = plot_roc_curve(test_y, test_scores, save_path=RESULTS_DIR / "roc_curve_densenet121_finetune.png")
plt.show()

## 11. Save Results

In [ ]:
def ns_to_dict(ns):
    """Convert SimpleNamespace (or None) to plain dict for JSON."""
    return vars(ns) if ns is not None else None

In [ ]:
from metrics import compute_accuracy, compute_roc_auc

save_results(
    model = f"DenseNet121-finetune(unfreeze_from={cfg.model.unfreeze_from})",
    img_size = list(IMG_SIZE),
    epochs = NUM_EPOCHS,
    batch_size = BATCH_SIZE,
    lr = HEAD_LR,  # campo único — backbone_lr vai no augmentation dict abaixo
    seed = SEED,
    pos_weight = pos_weight,
    best_val_auc = best_val_auc,
    augmentation = {
        **ns_to_dict(cfg.data.augmentations),
        "backbone_lr": BACKBONE_LR,
        "unfreeze_from": cfg.model.unfreeze_from,
    },
    test_accuracy = compute_accuracy(test_y, test_scores, threshold=best_threshold),
    test_auc_roc =  compute_roc_auc(test_y, test_scores),
    history = history,
    results_path = RESULTS_DIR / "densenet121_finetune_results.json"
)